# Powerflow Plotting Quick Test

This notebook directly tests `plot_powerflow_heatmap` on a step-4 output file.

Use the Python kernel from `GridExpand/4.powerflow/.venv` so all dependencies (including pandapower and plotly) are available.

In [ ]:
from pathlib import Path
import sys

# Resolve repository root even if notebook opens with a different working directory.
cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
root = next((p for p in candidates if (p / "GridExpand").exists()), None)
if root is None:
    raise RuntimeError("Could not find repository root containing 'GridExpand'.")

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

plotting_dir = root / "GridExpand/4.powerflow/plotting"
if str(plotting_dir) not in sys.path:
    sys.path.insert(0, str(plotting_dir))

root

In [ ]:
from IPython.display import clear_output, display
import ipywidgets as widgets
import numpy as np
import pandas as pd
import plotly.graph_objects as go

from powerflow_plotting import plot_powerflow_heatmap

h5_path = root / "GridExpand/4.powerflow/Output/900_80803_2_-1.h5"
stage = "pre"
if not h5_path.exists():
    raise FileNotFoundError(f"Output file not found: {h5_path}")

vm = pd.read_hdf(h5_path, key=f"/pwrflw/output/{stage}/vm")
transformer_import = pd.read_hdf(h5_path, key=f"/pwrflw/output/{stage}/demand_import")
transformer_import = transformer_import.copy()
transformer_import["s_mva"] = np.hypot(
    transformer_import["p_mw"],
    transformer_import["q_mvar"],
)
critical_timestep = int(transformer_import["s_mva"].idxmax())
n_timesteps = len(vm)

timeseries_fig = go.Figure()
timeseries_fig.add_trace(
    go.Scatter(
        x=transformer_import.index,
        y=transformer_import["p_mw"],
        mode="lines",
        name="P import [MW]",
    )
)
timeseries_fig.add_trace(
    go.Scatter(
        x=transformer_import.index,
        y=transformer_import["s_mva"],
        mode="lines",
        name="S import [MVA]",
    )
)
timeseries_fig.add_vline(
    x=critical_timestep,
    line_dash="dash",
    line_color="red",
    annotation_text=f"peak t={critical_timestep}",
    annotation_position="top right",
)
timeseries_fig.update_layout(
    title="Transformer Import Time Series",
    xaxis_title="Timestep",
    yaxis_title="Power",
    hovermode="x unified",
    legend={"orientation": "h", "y": 1.08},
    margin={"l": 60, "r": 30, "t": 70, "b": 50},
)
display(timeseries_fig)

timestep_slider = widgets.IntSlider(
    value=critical_timestep,
    min=0,
    max=n_timesteps - 1,
    step=1,
    description="Timestep",
    continuous_update=False,
    layout=widgets.Layout(width="600px"),
)
plot_output = widgets.Output()

def render_timestep(change=None):
    with plot_output:
        clear_output(wait=True)
        fig = plot_powerflow_heatmap(
            h5_path=h5_path,
            stage=stage,
            timestep=timestep_slider.value,
            on_map=False,
            map_style="light",
            cmap="Jet",
            climits_volt=(0.9, 1.1),
            climits_load=(0.0, 100.0),
            show_household_buses=False,
            show=False,
        )
        display(fig)

timestep_slider.observe(render_timestep, names="value")
display(timestep_slider, plot_output)
render_timestep()
